# 00 · Start hier

Welkom bij **Starten met IoT** (onderdeel van de les **Klimaatbeheersing in de veehouderij**). Dit notebook controleert of je werkomgeving klaar is en herhaalt de basisbegrippen voor de volgende onderdelen.

**Hoe werk je met een notebook?**
Een notebook bestaat uit tekstcellen en codecellen. Klik in een codecel en druk op **Shift + Enter** om de code uit te voeren. Voer de cellen van boven naar beneden uit.

> Werk je in GitHub Codespaces en vraagt VS Code om een *kernel*? Kies **Python 3.12** (of de hoogst beschikbare Python-versie).

## 1. Werkt Python en zijn de pakketten geïnstalleerd?

In [ ]:
import sys, platform
import paho.mqtt.client as mqtt
import requests, pandas as pd, matplotlib

print("Python      :", sys.version.split()[0], "op", platform.system())
print("paho-mqtt   :", mqtt.__version__ if hasattr(mqtt, "__version__") else "ok")
print("requests    :", requests.__version__)
print("pandas      :", pd.__version__)
print("matplotlib  :", matplotlib.__version__)
print("\nAlles geladen. Je kunt verder.")

## 2. Begrippen die je nodig hebt

| Begrip | Betekenis in dit practicum |
|---|---|
| **IoT-toestel / node / eindpunt** | Een sensor met een radio en een batterij, die zelfstandig metingen verstuurt. Voorbeeld: Dragino S31-LB. |
| **Verbindingstype** | De *fysieke* weg waarlangs bits reizen: wifi, ethernet, LoRaWAN, Bluetooth, NB-IoT/LTE-M. Vergelijk het met het type weg (fietspad, autosnelweg, spoor). |
| **Gegevensuitwisselingsprotocol** | De *afspraak* over hoe berichten eruitzien en wie wanneer spreekt: MQTT, HTTP, CoAP, Modbus. Vergelijk het met de taal en etiquette op die weg. |
| **Gateway** | Toestel dat tussen twee verbindingstypes vertaalt, bv. van LoRaWAN naar ethernet/internet. |
| **Broker** | De centrale "postbus" bij MQTT: toestellen publiceren berichten op een *topic*, andere toestellen abonneren zich op dat topic. |
| **Topic** | Een adres met schuine strepen, bv. `stal1/afdeling3/temperatuur`. |
| **Payload** | De inhoud van een bericht: ruwe bytes (hex), tekst of JSON. |
| **JSON** | Een gestructureerd tekstformaat met sleutels en waarden: `{"temperature": 21.4, "humidity": 63.2}`. |
| **Uplink / downlink** | Bericht van sensor naar server (uplink) of van server naar sensor (downlink). |

**Rode draad in dit practicum:** Een temperatuur- en vochtsensor in een stalafdeling verstuurt elke 10 minuten een meting, die uiteindelijk op het scherm van de veehouder of in de klimaatcomputer belandt.

## 3. Verbinding met de MQTT-broker testen

We gebruiken de publieke broker van HiveMQ (`broker.hivemq.com`). Aangezien deze broker voor iedereen toegankelijk is, stuur je hier nooit gevoelige of persoonlijke gegevens naartoe.

De onderstaande code verbindt met de broker, abonneert zich op een testtopic, verstuurt een bericht en controleert of het bericht correct ontvangen wordt.

In [ ]:
import paho.mqtt.client as mqtt
import time, uuid

BROKER = "broker.hivemq.com"
POORT  = 1883                       # standaard MQTT-poort (onversleuteld)
TOPIC  = f"ugent/i700301/test/{uuid.uuid4().hex[:8]}"   # uniek testtopic

ontvangen = []

def bij_bericht(client, userdata, msg):
    ontvangen.append(msg.payload.decode())

client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
client.on_message = bij_bericht
client.connect(BROKER, POORT, keepalive=60)
client.loop_start()
client.subscribe(TOPIC)
time.sleep(1)
client.publish(TOPIC, "hallo vanuit mijn notebook")

for _ in range(50):                  # max. 5 s wachten
    if ontvangen:
        break
    time.sleep(0.1)

client.loop_stop(); client.disconnect()
print("Topic   :", TOPIC)
print("Ontvangen:", ontvangen or "NIETS — zie de tip hieronder")

**Krijg je geen bericht of een time-out?**
Sommige netwerken (zoals campuswifi) blokkeren poort 1883. Gebruik in dat geval MQTT via **websockets** op poort 8884 met TLS. Vervang in de code hierboven de regels voor `client = mqtt.Client(...)` en `client.connect(...)` door:

```python
client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, transport="websockets")
client.tls_set()
client.ws_set_options(path="/mqtt")
client.connect("broker.hivemq.com", 8884, keepalive=60)
```

In GitHub Codespaces werkt poort 1883 normaal gezien direct.

## 4. Verbinding met een HTTP-API testen

Via een **HTTP-API** vraag je rechtstreeks gegevens op aan een webserver (meestal in JSON-formaat). Onderstaande code vraagt de actuele temperatuur in Gent op bij Open-Meteo.

In [ ]:
import requests

url = "https://api.open-meteo.com/v1/forecast"
parameters = {
    "latitude": 51.05, "longitude": 3.72,          # Gent
    "current": "temperature_2m",
}
antwoord = requests.get(url, params=parameters, timeout=10)
print("HTTP-statuscode:", antwoord.status_code, "(200 = OK)")

gegevens = antwoord.json()                          # JSON-tekst -> Python-dictionary
huidig = gegevens["current"]
print(f"Gent, {huidig['time']} UTC: {huidig['temperature_2m']} °C")

### Opdracht: Pas de API-oproep aan

Kopieer de code van de vorige cel naar de lege cel hieronder en pas ze stap voor stap aan. De benodigde parameternamen vind je in de [Open-Meteo documentatie](https://open-meteo.com/en/docs).

**Hoe gebruik je de documentatie?**
1. Vul bovenaan de gewenste **Latitude** en **Longitude** in.
2. Vink de gewenste variabelen aan onder **Current Weather** (voor actuele data) of **Hourly Weather Variables** (voor een tijdsreeks).
3. Scroll naar **API Response** en selecteer **Python**. De gegenereerde code toont precies de parameternamen die je in de dictionary `parameters` moet zetten.

*Tip:* Open-Meteo werkt op basis van een weermodel. Je geeft een willekeurige coördinaat op, en de API berekent de waarden voor het dichtstbijzijnde roosterpunt. Het antwoord bevat onder `latitude` en `longitude` de exacte locatie van dat roosterpunt. De bijbehorende eenheden vind je terug onder `current_units` en `hourly_units`.

#### Stappen:
1. **Relatieve vochtigheid in Gent:** Voeg de parameter voor relatieve vochtigheid toe aan `current` (gescheiden door een komma). Druk de temperatuur en vochtigheid af met hun eenheden uit `current_units`.
2. **Eigen locatie (actuele temperatuur):** Zoek via Google Maps de coördinaten van je thuisadres, stal of een andere locatie (rechtsklik op de kaart). Pas `latitude` en `longitude` aan en vraag de temperatuur op. Druk zowel de gevraagde coördinaten als de door de API gebruikte roostercoördinaten af.
3. **Alle actuele variabelen:** Vraag voor jouw locatie alle variabelen uit **Current Weather** op via `current`. Druk alle ontvangen waarden netjes af met hun eenheid.
4. **Tijdsreeks over meerdere dagen:** Vraag voor jouw locatie de temperatuur en vochtigheid op per uur. Gebruik hiervoor `hourly`, stel `past_days` in op minstens 1, `forecast_days` op minstens 2 en `timezone` op `"Europe/Brussels"`. Druk het totaal aantal uren af, alsook de waarden op het eerste en laatste tijdstip (uit `hourly`).

*(Onderaan vind je een ingeklapt voorbeeld dat stap 3 en 4 uitvoert voor Kortrijk.)*

In [ ]:
# Kopieer hier de code van de vorige cel en werk de 4 stappen uit.
# Documentatie & Python-voorbeeld: https://open-meteo.com/en/docs

# Stap 1: Gent - voeg relatieve vochtigheid toe aan 'current'

# Stap 2: Eigen locatie - pas latitude/longitude aan en vergelijk met het roosterpunt

# Stap 3: Eigen locatie - vraag alle 'Current Weather' variabelen op

# Stap 4: Eigen locatie - vraag 'hourly' data op over meerdere dagen (past_days, forecast_days, timezone)


In [ ]:
# Voorbeeld — Kortrijk. Zelf vervang je de coördinaten door die van je eigen plek.
import requests

url = "https://api.open-meteo.com/v1/forecast"
gevraagd_lat, gevraagd_lon = 50.828, 3.265

parameters = {
    "latitude": gevraagd_lat,
    "longitude": gevraagd_lon,
    "current": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature",
        "is_day",
        "snowfall",
        "weather_code",
        "cloud_cover",
        "pressure_msl",
        "surface_pressure",
        "wind_speed_10m",
        "wind_direction_10m",
        "wind_gusts_10m",
    ]),
    "cell_selection": "nearest",
}
antwoord = requests.get(url, params=parameters, timeout=10)
antwoord.raise_for_status()
gegevens = antwoord.json()
huidig, eenheden = gegevens["current"], gegevens["current_units"]

print(f"Gevraagd:    {gevraagd_lat}, {gevraagd_lon}")
print(f"Roosterpunt: {gegevens['latitude']}, {gegevens['longitude']}")
print(huidig["time"])
for sleutel, waarde in huidig.items():
    if sleutel in ("time", "interval"):
        continue
    print(f"{sleutel}: {waarde} {eenheden[sleutel]}")

reeks = requests.get(url, params={
    "latitude": gevraagd_lat,
    "longitude": gevraagd_lon,
    "hourly": "temperature_2m,relative_humidity_2m",
    "past_days": 1,
    "forecast_days": 2,
    "timezone": "Europe/Brussels",
    "cell_selection": "nearest",
}, timeout=10)
reeks.raise_for_status()
uurlijks = reeks.json()["hourly"]
eenheid = reeks.json()["hourly_units"]
tijden = uurlijks["time"]
print(f"{len(tijden)} tijdstappen, {tijden[0]} → {tijden[-1]}")
for i in (0, -1):
    print(f"{tijden[i]}: {uurlijks['temperature_2m'][i]} {eenheid['temperature_2m']}, "
          f"{uurlijks['relative_humidity_2m'][i]} {eenheid['relative_humidity_2m']}")


## 5. Klaar

Lopen alle tests en de opdracht naar behoren? Dan kun je verder naar `01_sensor_kiezen_en_verbinden.ipynb`.

**Belangrijkste verschil om te onthouden:**

* **MQTT:** Je maakt één verbinding. De broker stuurt (publish) nieuwe berichten automatisch door naar geabonneerde clients (subscribe).
* **HTTP:** Je stuurt voor elke meting een expliciet verzoek (request) en ontvangt één antwoord (response). Welke gegevens de server teruggeeft, bepaal je via de parameters in het verzoek.